# Chinook Music Store: Revenue & Customer Analytics
 Data Export and SQL analysis for Tableau Exporting analytical views for visualization in Tableau.

## Importing libraries and data loading

In [2]:
!pip install duckdb
import duckdb
conn = duckdb.connect()

from google.colab import drive
drive.mount('/content/drive')

import os
path = '/content/drive/MyDrive/portfolio projects/Chinook/data/'
for file in os.listdir(path):
    if file.endswith('.csv'):
        table_name = file.replace('.csv', '').lower()
        conn.execute(f"CREATE TABLE {table_name} AS SELECT * FROM read_csv_auto('{path}{file}')")

conn.execute("SHOW TABLES").fetchdf()

Mounted at /content/drive


,name
0,album
1,artist
2,customer
3,employee
4,genre
5,invoice
6,invoiceline
7,mediatype
8,playlist
9,playlisttrack


## Data quality checking

In [3]:
# review Invoice table
conn.execute("DESCRIBE Invoice").fetchdf()

#numbers and dates recognized correctly

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(InvoiceId) AS invoice_id
        , COUNT(CustomerId) AS customer_id
        , COUNT(InvoiceDate) AS invoice_date
        , COUNT(Total) AS total
    FROM invoice
""").fetchdf()

# null isnt counded, so if we have some null - count rows number will be lower then general rows number
# here we have no null

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT InvoiceId) AS unique_invoices
    FROM invoice
""").fetchdf()
# we have no dublicates (invoiceid)

,total_rows,unique_invoices
0,412,412


In [4]:
# review Customer table
conn.execute("DESCRIBE Customer").fetchdf()

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(CustomerId) AS customerId
        , COUNT(Country) AS country
        , COUNT(City) AS city
        , COUNT(SupportRepId) AS supportRepId
    FROM Customer
""").fetchdf()

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(DISTINCT CustomerId) AS unique_invoices
    FROM Customer
""").fetchdf()
# we have no dublicates (CustomerId)

,total_rows,unique_invoices
0,59,59


In [5]:
conn.execute("DESCRIBE Artist").fetchdf()
# all data recognized right

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(ArtistId)
        , COUNT(Name)
    FROM Artist
""").fetchdf()
# here are 275 lines, no empty values

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT ArtistId)
    FROM Artist
""").fetchdf()
# here are no dublicates (ArtistId)

,total_rows,count(DISTINCT ArtistId)
0,275,275


In [6]:
conn.execute("DESCRIBE Album").fetchdf()
# all data recognized right

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(AlbumId)
        , COUNT(Title)
        , COUNT(ArtistId)
    FROM Album
""").fetchdf()
# here are 347 lines, no empty values

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT AlbumId)
    FROM Album
""").fetchdf()
# here are no dublicates (AlbumId)

,total_rows,count(DISTINCT AlbumId)
0,347,347


In [7]:
conn.execute("DESCRIBE Track").fetchdf()
# all data recognized right

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(TrackId)
        , COUNT(AlbumId)
    FROM Track
""").fetchdf()
# here are 3503 lines, no empty values

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT TrackId)
    FROM Track
""").fetchdf()
# here are no dublicates (TrackId)

,total_rows,count(DISTINCT TrackId)
0,3503,3503


In [8]:
conn.execute("DESCRIBE InvoiceLine").fetchdf()
# all data recognized right

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(InvoiceLineId)
        , COUNT(InvoiceId)
        , COUNT(TrackId)
        , COUNT(UnitPrice)
        , COUNT(Quantity)
    FROM InvoiceLine
""").fetchdf()
# here are 2240 lines, no empty values

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT InvoiceLineId)
    FROM InvoiceLine
""").fetchdf()
# here are no dublicates (InvoiceLineId)

,total_rows,count(DISTINCT InvoiceLineId)
0,2240,2240


In [9]:
conn.execute("DESCRIBE Employee").fetchdf()
# all data recognized right

conn.execute("""
    SELECT
        COUNT(*) AS total_rows
        , COUNT(EmployeeId)
        , COUNT(Country)
    FROM Employee
""").fetchdf()
# here are 8 lines, no empty values

conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT EmployeeId)
    FROM Employee
""").fetchdf()
# here are no dublicates (EmployeeId)

,total_rows,count(DISTINCT EmployeeId)
0,8,8


## 1. Revenue performance analysis
### Monthly revenue, invoice count, and average order value across 2009–2013. Identifies sales trends and month-over-month dynamics.

In [10]:
conn.execute("""
CREATE OR REPLACE VIEW revenue_analysis AS
SELECT
  STRFTIME(InvoiceDate, '%Y-%m') AS year_month
  , SUM(Total) AS monthly_revenue
  , COUNT(InvoiceId) AS invoice_count
  , ROUND(AVG(Total), 2) AS average_order_value
FROM invoice
GROUP BY 1
ORDER BY year_month ASC
""")

conn.execute("SELECT * FROM revenue_analysis").fetchdf()

,year_month,monthly_revenue,invoice_count,average_order_value
0,2009-01,35.64,6,5.94
1,2009-02,37.62,7,5.37
2,2009-03,37.62,7,5.37
3,2009-04,37.62,7,5.37
4,2009-05,37.62,7,5.37
5,2009-06,37.62,7,5.37
6,2009-07,37.62,7,5.37
7,2009-08,37.62,7,5.37
8,2009-09,37.62,7,5.37
9,2009-10,37.62,7,5.37


## Summary:
Overall Sales Performance & Trends (2009–2013):
During the analyzed period, the store generated \$2,329 in total revenue across 412 invoices, driven by a base of 59 unique customers. Revenue distribution remained relatively uniform over time, maintaining a stable sales flow. The highest earnings were recorded in Q2 2011 (\$145, 21 invoices), Q1 2010 (\$144, 21 invoices), and Q3 2012 (\$134, 20 invoices). The lowest performing quarter was Q4 2011 (\$99, 20 invoices). Despite these minor revenue fluctuations, the transaction volume remained consistent.

## 2. Customer value analysis
### Revenue per customer, spend ranking, and Pareto analysis to identify the top customers driving 80% of total revenue.

2.1 Customer Segmentation

In [11]:
conn.execute("""
CREATE OR REPLACE VIEW customer_segmentation AS
WITH customer_summary AS (
SELECT c.CustomerId
  , COUNT (InvoiceId) as invoice_per_customer
  , COALESCE(SUM(Total), 0) as total_spend
  , ROUND(AVG(Total), 2) AS average_order_value
  , Country

  FROM Customer c
  LEFT JOIN Invoice i
    ON c.CustomerId = i.CustomerId
  GROUP BY c.CustomerId, Country
  )

 SELECT *
    , NTILE(4) OVER(ORDER BY total_spend DESC) AS spend_quartile
 FROM customer_summary
""")

conn.execute("SELECT * FROM customer_segmentation ORDER BY total_spend DESC").fetchdf()


,CustomerId,invoice_per_customer,total_spend,average_order_value,Country,spend_quartile
0,6,7,49.62,7.09,Czech Republic,1
1,26,7,47.62,6.80,USA,1
2,57,7,46.62,6.66,Chile,1
3,46,7,45.62,6.52,Ireland,1
4,45,7,45.62,6.52,Hungary,1
5,28,7,43.62,6.23,USA,1
6,37,7,43.62,6.23,Germany,1
7,24,7,43.62,6.23,USA,1
8,7,7,42.62,6.09,Austria,1
9,25,7,42.62,6.09,USA,1


## Summary:
All 59 customers made 6-7 invoices each, making transaction-count segmentation meaningless. Total spend ranged narrowly from \$36.64 to \$49.62, with the vast majority clustered at \$37.62. NTILE(4) was applied on total spend to create quartiles, but due to the highly uniform distribution, quartile boundaries are arbitrary — many customers with identical spend (\$37.62) fall into different groups. This reflects the artificial nature of the Chinook dataset rather than real purchasing behavior.

2.2 Pareto Analysis

In [12]:
conn.execute("""
CREATE OR REPLACE VIEW pareto_analysis AS
WITH customer_summary AS (
SELECT c.CustomerId
  , COUNT (InvoiceId) as invoice_per_customer
  , COALESCE(SUM(Total), 0) as total_spend
  , Country
  FROM Customer c
  LEFT JOIN Invoice i
    ON c.CustomerId = i.CustomerId
  GROUP BY c.CustomerId, Country
  ),

    customer_pareto AS (
    SELECT *
        , SUM(total_spend) OVER(
            ORDER BY total_spend DESC, CustomerId DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
          ) AS cumulative_revenue
        , SUM(total_spend) OVER() AS total_revenue
    FROM customer_summary
)


SELECT *
    , ROUND(cumulative_revenue * 100.0 / total_revenue, 2) AS cumulative_pct
FROM customer_pareto
""")

conn.execute("SELECT * FROM pareto_analysis ORDER BY total_spend DESC, CustomerId DESC").fetchdf()

conn.execute("SELECT COUNT(*) FROM pareto_analysis WHERE cumulative_pct <= 80").fetchdf()
# 46 out of 59 customers (78%) generate 80% revenue.
# This is not a classic 20/80 - due to an artificial dataset where all customers have almost the same spend.
# It is important to note this in the README as a limitation of the dataset.


,count_star()
0,46


## Summary:
Pareto analysis shows that 46 out of 59 customers (78%) generate 80% of total revenue — significantly flatter than the classic 80/20 rule. This is a direct consequence of the artificial dataset: spend is nearly uniform across customers (\$37.62–\$49.62), so no small group of high-value customers stands out.

## 3. Geography Performance Revenue
### Identifies key markets and revenue concentration across countries.

In [13]:
conn.execute("""
CREATE OR REPLACE VIEW geography_analysis AS
  SELECT
    Country
    , ROUND(COALESCE(SUM(Total), 0), 2) as total_spend
    , COUNT(DISTINCT c.CustomerId) as customer_count
    , COUNT(InvoiceId) AS invoice_count
    , ROUND(AVG(Total), 2) as avg_invoice
    , ROUND (SUM(Total) / COUNT(DISTINCT c.CustomerId), 2) as revenue_per_customer
    , ROUND(SUM(Total) * 100.0 / SUM(SUM(Total)) OVER(), 2) AS revenue_share
  FROM Customer c
  LEFT JOIN Invoice i
    ON c.CustomerId = i.CustomerId
  GROUP BY c.Country
  """)

conn.execute("SELECT * FROM geography_analysis ORDER BY total_spend DESC").fetchdf()

,Country,total_spend,customer_count,invoice_count,avg_invoice,revenue_per_customer,revenue_share
0,USA,523.06,13,91,5.75,40.24,22.46
1,Canada,303.96,8,56,5.43,37.99,13.05
2,France,195.10,5,35,5.57,39.02,8.38
3,Brazil,190.10,5,35,5.43,38.02,8.16
4,Germany,156.48,4,28,5.59,39.12,6.72
5,United Kingdom,112.86,3,21,5.37,37.62,4.85
6,Czech Republic,90.24,2,14,6.45,45.12,3.88
7,Portugal,77.24,2,14,5.52,38.62,3.32
8,India,75.26,2,13,5.79,37.63,3.23
9,Chile,46.62,1,7,6.66,46.62,2.00


### Summary:
The USA leads by a wide margin with \$523.06 in total revenue (22.5% of global sales) and 13 customers — the largest customer base. Canada and France follow with 13% and 8.4% respectively. The top 5 countries (USA, Canada, France, Brazil, Germany) together account for ~59% of revenue, largely reflecting customer count rather than per-customer spend. Single-customer countries like Chile (\$46.62) and Hungary (\$45.62) show higher revenue-per-customer than multi-customer markets, suggesting AOV varies more by individual than by geography.


## 4. Product performance
### Analyzing revenue by track, album, artist, and genre to identify top-performing content across the catalog.

In [14]:
conn.execute("""
SELECT
  ar.ArtistId
  , AlbumId
FROM Artist ar
  LEFT JOIN Album al
    ON ar.ArtistId = al.ArtistId
  """).fetchdf()

conn.execute("""
SELECT
    COUNT(*) AS total_artists
    , COUNT(AlbumId) AS with_albums
    , COUNT(*) - COUNT(AlbumId) AS without_albums
FROM Artist ar
LEFT JOIN Album al
    ON ar.ArtistId = al.ArtistId
  """).fetchdf()
  # 418 artists in total, 347 have albums, 71 without any
  # 71 artists have no albums and do not appear in sales

conn.execute("""
  SELECT
    COUNT(*) AS total_tracks,
    COUNT(AlbumId) AS with_album,
    COUNT(*) - COUNT(AlbumId) AS without_album
FROM Track
""").fetchdf()
# there are no tracks without album

,total_tracks,with_album,without_album
0,3503,3503,0


In [15]:
conn.execute("""
CREATE OR REPLACE VIEW sales_analysis as
SELECT
  ar.ArtistId as ArtistId
  , ar.Name as artist_name
  , al.AlbumId
  , al.Title as album_name

  , tr.TrackId
  , tr.Name as track_name
  , tr.MediaTypeId


  , il.UnitPrice as track_price
  , il.Quantity as track_quantity

  , g.Name as genre_name

  , (il.UnitPrice * il.Quantity) AS revenue

  , i.InvoiceDate
FROM Artist ar
  INNER JOIN Album al
    ON ar.ArtistId = al.ArtistId

  INNER JOIN Track tr
    ON al.AlbumId = tr.AlbumId

  INNER JOIN InvoiceLine il
    ON tr.TrackId = il.TrackId

  INNER JOIN Genre g
    ON tr.GenreId = g.GenreId

  INNER JOIN Invoice i
    ON il.InvoiceId = i.InvoiceId
  """)
conn.execute("SELECT * FROM sales_analysis").fetchdf()


,ArtistId,artist_name,AlbumId,album_name,TrackId,track_name,MediaTypeId,track_price,track_quantity,genre_name,revenue,InvoiceDate
0,1,AC/DC,1,For Those About To Rock We Salute You,1,For Those About To Rock (We Salute You),1,0.99,1,Rock,0.99,2010-04-13
1,2,Accept,2,Balls to the Wall,2,Balls to the Wall,2,0.99,1,Rock,0.99,2011-07-25
2,2,Accept,3,Restless and Wild,3,Fast As a Shark,2,0.99,1,Rock,0.99,2012-11-01
3,2,Accept,3,Restless and Wild,4,Restless and Wild,2,0.99,1,Rock,0.99,2009-01-01
4,2,Accept,3,Restless and Wild,5,Princess of the Dawn,2,0.99,1,Rock,0.99,2010-04-13
...,...,...,...,...,...,...,...,...,...,...,...,...
2235,245,Michael Tilson Thomas & San Francisco Symphony,312,Berlioz: Symphonie Fantastique,3446,"Symphonie Fantastique, Op. 14: V. Songe d'une ...",2,0.99,1,Classical,0.99,2010-03-21
2236,252,Amy Winehouse,321,Back to Black,3455,Rehab,2,0.99,1,R&B/Soul,0.99,2010-03-21
2237,257,"Academy of St. Martin in the Fields, Sir Nevil...",327,Bach: Orchestral Suites Nos. 1 - 4,3482,"Suite No. 3 in D, BWV 1068: III. Gavotte I & II",2,0.99,1,Classical,0.99,2010-04-11
2238,263,"Equale Brass Ensemble, John Eliot Gardiner & M...",333,Purcell: Music for the Queen Mary,3488,"Music for the Funeral of Queen Mary: VI. ""Thou...",2,0.99,1,Classical,0.99,2010-04-12


## Summary:
This view serves as the core fact table for sales analysis, joining artists, albums, tracks, invoice lines, and genres into a single structure. It covers 2,240 sold track records with revenue calculated as unit price × quantity. The view underpins downstream analysis of top artists, albums, genres, and revenue contribution.


## 5. Sales support impact
### Revenue, customer count, and average spend per support representative. Evaluates workload distribution and individual contribution to total sales.

In [16]:
conn.execute("""
CREATE OR REPLACE VIEW employee_analysis as
SELECT
  em.EmployeeId
  , em.FirstName || ' ' || em.LastName AS support_name
  , em.Country as support_country

  , COUNT(DISTINCT c.CustomerId) as served_client_numb

  , COUNT(DISTINCT InvoiceId) as completed_inv
  , COALESCE(SUM(i.Total), 0) as revenue_per_emp

  , COALESCE(ROUND(SUM(i.Total) / NULLIF(COUNT(DISTINCT c.CustomerId), 0), 2), 0) as avg_revenue_per_customer
FROM Employee em
  LEFT JOIN Customer c
    ON em.EmployeeId = c.SupportRepId

  LEFT JOIN Invoice i
    ON c.CustomerId = i.CustomerId

  GROUP BY EmployeeId, support_name, support_country
  ORDER BY revenue_per_emp DESC

""")

conn.execute("SELECT * FROM employee_analysis").fetchdf()

,EmployeeId,support_name,support_country,served_client_numb,completed_inv,revenue_per_emp,avg_revenue_per_customer
0,3,Jane Peacock,Canada,21,146,833.04,39.67
1,4,Margaret Park,Canada,20,140,775.40,38.77
2,5,Steve Johnson,Canada,18,126,720.16,40.01
3,7,Robert King,Canada,0,0,0.00,0.00
4,6,Michael Mitchell,Canada,0,0,0.00,0.00
5,2,Nancy Edwards,Canada,0,0,0.00,0.00
6,8,Laura Callahan,Canada,0,0,0.00,0.00
7,1,Andrew Adams,Canada,0,0,0.00,0.00


## Summary:
Only 3 out of 8 employees acted as customer support reps. Jane Peacock led with 21 clients and 833 in revenue, followed by Margaret Park (20 clients, \$775) and Steve Johnson (18 clients, \$720). Average revenue per customer was nearly identical across all three (\$38.77–\$40.01), suggesting performance differences are driven by client volume rather than upselling ability.

In [17]:
import os
output_path = '/content/drive/MyDrive/portfolio projects/Chinook/data/tableau/'
os.makedirs(output_path, exist_ok=True)

views = ['revenue_analysis', 'pareto_analysis', 'geography_analysis', 'sales_analysis', 'employee_analysis']

for view in views:
    conn.execute(f"COPY (SELECT * FROM {view}) TO '{output_path}{view}.csv' (HEADER, DELIMITER ',')")

print("Done")

Done
